# 04.1 - XGBoost Optuna tuning nhe tren subset

Notebook nay tune XGBoost thong minh hon grid search nhung tiet kiem compute:

1. Load `processed/clinvar_tabular_cnn_aligned.parquet`.
2. Tao feature nhu notebook 04.
3. Split train/val/test co dinh.
4. Lay subset stratified tu train de tune.
5. Optuna optimize PR-AUC tren tune validation.
6. Train final model tren full train voi best params.
7. Evaluate tren test set mot lan.

Khong nested CV, khong grid search.

In [13]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

PROJECT_DIR = Path("/mnt/MyData/Bioinformatics/Project")
PROCESSED_DIR = PROJECT_DIR / "processed"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

TABULAR_PATH = PROCESSED_DIR / "clinvar_tabular_cnn_aligned.parquet"
if not TABULAR_PATH.exists():
    raise FileNotFoundError(TABULAR_PATH)

pd.set_option("display.max_columns", 120)
TABULAR_PATH

PosixPath('/mnt/MyData/Bioinformatics/Project/processed/clinvar_tabular_cnn_aligned.parquet')

## 1. Cau hinh tuning

Mac dinh khong dung GPU. Voi may 6GB VRAM, CPU `tree_method="hist"` an toan hon.

In [14]:
RANDOM_STATE = 42
USE_GPU = True

# FEATURE_SET:
# - "biology_light": it shortcut tu metadata ClinVar, dung de kiem tra leakage/overfit tot hon.
# - "clinvar_metadata": dung them review/status/count metadata, co the cho diem cao hon nhung de bi shortcut.
FEATURE_SET = "biology_light"

# SPLIT_MODE:
# - "gene_group": train/val/test tach GeneSymbol, giam leak qua cac variant cung gene.
# - "random": split ngau nhien theo row, nhanh nhung optimistic hon.
SPLIT_MODE = "gene_group"

TUNE_SAMPLE_SIZE = 20_000
N_TRIALS = 15
TIMEOUT_SECONDS = 30 * 60

FINAL_N_ESTIMATORS = 1_200

# Doi thanh 1.0 de final train tren full train. Co the giam de smoke test.
FINAL_TRAIN_FRACTION = 1.0

FEATURE_SET, SPLIT_MODE, TUNE_SAMPLE_SIZE, N_TRIALS, TIMEOUT_SECONDS, FINAL_N_ESTIMATORS


('biology_light', 'gene_group', 20000, 15, 1800, 1200)

## 2. Load tabular dataset va optional annotation

In [15]:
df = pd.read_parquet(TABULAR_PATH).copy()
y = df["Y"].to_numpy(dtype=np.int8)

ANNOTATION_CANDIDATES = [
    PROCESSED_DIR / "clinvar_external_annotations.parquet",
    PROJECT_DIR / "Data" / "annotations" / "variant_annotations.parquet",
    PROJECT_DIR / "Data" / "annotations" / "variant_annotations.csv",
]
variant_key_cols = ["Chromosome", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF"]
annotation_path = next((path for path in ANNOTATION_CANDIDATES if path.exists()), None)

if annotation_path is not None:
    print(f"Loading annotation file: {annotation_path}")
    annotation_df = pd.read_parquet(annotation_path) if annotation_path.suffix == ".parquet" else pd.read_csv(annotation_path)
    available_annotation_cols = [
        col for col in ["consequence", "gnomAD_AF", "CADD_PHRED", "SpliceAI_max"]
        if col in annotation_df.columns
    ]
    annotation_df = annotation_df[variant_key_cols + available_annotation_cols].copy()
    for col in variant_key_cols:
        annotation_df[col] = annotation_df[col].astype(str)
        df[col] = df[col].astype(str)
    annotation_df = annotation_df.drop_duplicates(subset=variant_key_cols, keep="first")
    df = df.merge(annotation_df, on=variant_key_cols, how="left")
else:
    print("No annotation file found; using placeholders.")

if "consequence" not in df.columns:
    df["consequence"] = "unknown"
else:
    df["consequence"] = df["consequence"].fillna("unknown").astype(str)

for col in ["gnomAD_AF", "CADD_PHRED", "SpliceAI_max"]:
    if col not in df.columns:
        df[col] = np.nan
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.shape)
print(pd.Series(y).value_counts())
print({
    "consequence_known": int(df["consequence"].ne("unknown").sum()),
    "gnomAD_AF_non_null": int(df["gnomAD_AF"].notna().sum()),
    "CADD_PHRED_non_null": int(df["CADD_PHRED"].notna().sum()),
})

Loading annotation file: /mnt/MyData/Bioinformatics/Project/Data/annotations/variant_annotations.parquet
(460488, 59)
0    401054
1     59434
Name: count, dtype: int64
{'consequence_known': 0, 'gnomAD_AF_non_null': 0, 'CADD_PHRED_non_null': 0}


## 3. Feature set

Khong dung `ClinicalSignificance`, `ClinSigSimple`, `Y` lam feature.

In [16]:
feature_df = pd.DataFrame(index=df.index)
feature_df["Chromosome"] = df["Chromosome"].astype(str)
feature_df["ReferenceAlleleVCF"] = df["ReferenceAlleleVCF"].astype(str)
feature_df["AlternateAlleleVCF"] = df["AlternateAlleleVCF"].astype(str)
feature_df["ref_alt"] = df["ref_alt"].astype(str)
feature_df["consequence"] = df["consequence"].fillna("unknown").astype(str)

biology_numeric_cols = [
    "PositionVCF_int",
    "is_transition",
    "is_transversion",
    "variant_span",
    "chromosome_is_autosome",
    "chromosome_is_sex",
    "chromosome_is_mt",
]

clinvar_metadata_categorical_cols = [
    "ReviewStatusSimple",
    "OriginSimple",
]
clinvar_metadata_numeric_cols = [
    "NumberSubmitters_int",
    "SubmitterCategories_int",
    "has_rs",
    "tested_in_gtr_flag",
    "LastEvaluated_year",
    "phenotype_count",
    "rcv_count",
    "scv_count",
    "has_omim_other_id",
]

categorical_cols = [
    "Chromosome",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
    "ref_alt",
    "consequence",
]
numeric_source_cols = biology_numeric_cols.copy()

if FEATURE_SET == "clinvar_metadata":
    feature_df["ReviewStatusSimple"] = df["ReviewStatusSimple"].astype(str)
    feature_df["OriginSimple"] = df["OriginSimple"].fillna("unknown").astype(str)
    categorical_cols += clinvar_metadata_categorical_cols
    numeric_source_cols += clinvar_metadata_numeric_cols
elif FEATURE_SET != "biology_light":
    raise ValueError(f"FEATURE_SET khong hop le: {FEATURE_SET}")

for col in numeric_source_cols:
    feature_df[col] = pd.to_numeric(df[col], errors="coerce").fillna(-1).astype("float32")

for source_col in ["gnomAD_AF", "CADD_PHRED", "SpliceAI_max"]:
    feature_df[f"{source_col}_missing"] = df[source_col].isna().astype("int8")
    feature_df[f"{source_col}_filled"] = df[source_col].fillna(-1).astype("float32")

numeric_cols = [col for col in feature_df.columns if col not in categorical_cols]

forbidden_feature_cols = {
    "ClinicalSignificance", "ClinSigSimple", "Y", "Name", "VariationID", "#AlleleID",
    "PhenotypeIDS", "PhenotypeList", "RCVaccession", "SCVsForAggregateGermlineClassification",
}
leaked_cols = sorted(forbidden_feature_cols.intersection(feature_df.columns))
assert not leaked_cols, f"Leak columns in feature_df: {leaked_cols}"

print("FEATURE_SET:", FEATURE_SET)
print(feature_df.shape)
display(feature_df.head())
display(feature_df[categorical_cols].nunique())
print("categorical_cols:", categorical_cols)
print("numeric_cols:", numeric_cols)


FEATURE_SET: biology_light
(460488, 18)


,Chromosome,ReferenceAlleleVCF,AlternateAlleleVCF,ref_alt,consequence,PositionVCF_int,is_transition,is_transversion,variant_span,chromosome_is_autosome,chromosome_is_sex,chromosome_is_mt,gnomAD_AF_missing,gnomAD_AF_filled,CADD_PHRED_missing,CADD_PHRED_filled,SpliceAI_max_missing,SpliceAI_max_filled
0,20,G,A,G>A,unknown,25302322.0,1.0,0.0,1.0,1.0,0.0,0.0,1,-1.0,1,-1.0,1,-1.0
1,10,G,T,G>T,unknown,97611536.0,0.0,1.0,1.0,1.0,0.0,0.0,1,-1.0,1,-1.0,1,-1.0
2,10,C,T,C>T,unknown,97598848.0,1.0,0.0,1.0,1.0,0.0,0.0,1,-1.0,1,-1.0,1,-1.0
3,10,G,C,G>C,unknown,97584912.0,0.0,1.0,1.0,1.0,0.0,0.0,1,-1.0,1,-1.0,1,-1.0
4,10,T,G,T>G,unknown,97601928.0,0.0,1.0,1.0,1.0,0.0,0.0,1,-1.0,1,-1.0,1,-1.0


Chromosome            10
ReferenceAlleleVCF     4
AlternateAlleleVCF     4
ref_alt               16
consequence            1
dtype: int64

categorical_cols: ['Chromosome', 'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'ref_alt', 'consequence']
numeric_cols: ['PositionVCF_int', 'is_transition', 'is_transversion', 'variant_span', 'chromosome_is_autosome', 'chromosome_is_sex', 'chromosome_is_mt', 'gnomAD_AF_missing', 'gnomAD_AF_filled', 'CADD_PHRED_missing', 'CADD_PHRED_filled', 'SpliceAI_max_missing', 'SpliceAI_max_filled']


## 4. Split train/validation/test

In [17]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split

indices = np.arange(len(y))

if SPLIT_MODE == "random":
    train_idx, temp_idx = train_test_split(indices, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=RANDOM_STATE, stratify=y[temp_idx])
elif SPLIT_MODE == "gene_group":
    groups = df["GeneSymbol"].fillna("unknown").astype(str).to_numpy()
    unknown_mask = groups == "unknown"
    groups[unknown_mask] = np.array([f"unknown_{i}" for i in np.where(unknown_mask)[0]])

    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
    train_idx, temp_idx = next(gss1.split(indices, y, groups=groups))

    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
    rel_val_idx, rel_test_idx = next(gss2.split(temp_idx, y[temp_idx], groups=groups[temp_idx]))
    val_idx = temp_idx[rel_val_idx]
    test_idx = temp_idx[rel_test_idx]
else:
    raise ValueError(f"SPLIT_MODE khong hop le: {SPLIT_MODE}")

if FINAL_TRAIN_FRACTION < 1.0:
    train_idx, _ = train_test_split(
        train_idx,
        train_size=FINAL_TRAIN_FRACTION,
        random_state=RANDOM_STATE,
        stratify=y[train_idx],
    )

print("SPLIT_MODE:", SPLIT_MODE)
print(f"train: {len(train_idx):,}, labels={dict(zip(*np.unique(y[train_idx], return_counts=True)))}")
print(f"val:   {len(val_idx):,}, labels={dict(zip(*np.unique(y[val_idx], return_counts=True)))}")
print(f"test:  {len(test_idx):,}, labels={dict(zip(*np.unique(y[test_idx], return_counts=True)))}")

variant_key_cols = ["Chromosome", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF"]
train_keys = set(map(tuple, df.iloc[train_idx][variant_key_cols].astype(str).to_numpy()))
val_keys = set(map(tuple, df.iloc[val_idx][variant_key_cols].astype(str).to_numpy()))
test_keys = set(map(tuple, df.iloc[test_idx][variant_key_cols].astype(str).to_numpy()))
print("variant key overlap train/val:", len(train_keys & val_keys))
print("variant key overlap train/test:", len(train_keys & test_keys))

train_genes = set(df.iloc[train_idx]["GeneSymbol"].fillna("unknown").astype(str))
val_genes = set(df.iloc[val_idx]["GeneSymbol"].fillna("unknown").astype(str))
test_genes = set(df.iloc[test_idx]["GeneSymbol"].fillna("unknown").astype(str))
print("gene overlap train/val:", len((train_genes & val_genes) - {"unknown"}))
print("gene overlap train/test:", len((train_genes & test_genes) - {"unknown"}))


SPLIT_MODE: gene_group
train: 319,608, labels={np.int8(0): np.int64(278577), np.int8(1): np.int64(41031)}
val:   72,308, labels={np.int8(0): np.int64(62401), np.int8(1): np.int64(9907)}
test:  68,572, labels={np.int8(0): np.int64(60076), np.int8(1): np.int64(8496)}
variant key overlap train/val: 0
variant key overlap train/test: 0
gene overlap train/val: 0
gene overlap train/test: 0


## 5. Preprocess categorical once

Encode categorical feature mot lan roi reuse cho tuning de tranh fit Pipeline lap lai qua nhieu trials.

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from scipy import sparse

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

X_train_raw = feature_df.iloc[train_idx]
X_val_raw = feature_df.iloc[val_idx]
X_test_raw = feature_df.iloc[test_idx]
y_train = y[train_idx]
y_val = y[val_idx]
y_test = y[test_idx]

start = time.time()
X_train_all = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)
print(f"Preprocess time: {(time.time() - start) / 60:.2f} minutes")
print(X_train_all.shape, X_val.shape, X_test.shape)
print(type(X_train_all), X_train_all.dtype)

Preprocess time: 0.01 minutes
(319608, 48) (72308, 48) (68572, 48)
<class 'scipy.sparse._csr.csr_matrix'> float64


## 6. Tao tune subset stratified tu train

In [19]:
tune_size = min(TUNE_SAMPLE_SIZE, len(train_idx))
train_positions = np.arange(len(train_idx))

tune_positions, _ = train_test_split(
    train_positions,
    train_size=tune_size,
    random_state=RANDOM_STATE,
    stratify=y_train,
)

tune_train_positions, tune_val_positions = train_test_split(
    tune_positions,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_train[tune_positions],
)

X_tune_train = X_train_all[tune_train_positions]
y_tune_train = y_train[tune_train_positions]
X_tune_val = X_train_all[tune_val_positions]
y_tune_val = y_train[tune_val_positions]

print(f"tune_train: {X_tune_train.shape}, labels={dict(zip(*np.unique(y_tune_train, return_counts=True)))}")
print(f"tune_val:   {X_tune_val.shape}, labels={dict(zip(*np.unique(y_tune_val, return_counts=True)))}")

tune_train: (15000, 48), labels={np.int8(0): np.int64(13074), np.int8(1): np.int64(1926)}
tune_val:   (5000, 48), labels={np.int8(0): np.int64(4358), np.int8(1): np.int64(642)}


## 7. Optuna objective: optimize PR-AUC

In [20]:
import optuna
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

base_params = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
}
if USE_GPU:
    base_params["device"] = "cuda"


def objective(trial):
    params = {
        **base_params,
        "n_estimators": 800,
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.15, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 30.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "max_bin": trial.suggest_categorical("max_bin", [128, 256]),
    }
    model = XGBClassifier(**params)
    model.fit(
        X_tune_train,
        y_tune_train,
        eval_set=[(X_tune_val, y_tune_val)],
        verbose=False,
    )
    proba = model.predict_proba(X_tune_val)[:, 1]
    return average_precision_score(y_tune_val, proba)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SECONDS, show_progress_bar=True)

print("Best PR-AUC:", study.best_value)
print("Best params:")
study.best_params

[I 2026-05-10 09:21:57,001] A new study created in memory with name: no-name-8e6a6d35-3a82-4b32-bb63-af3c6ddabdd2


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-05-10 09:21:57,841] Trial 0 finished with value: 0.2916862987004084 and parameters: {'max_depth': 4, 'min_child_weight': 17.25471657328035, 'learning_rate': 0.08092546450005342, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'reg_lambda': 0.004993515273965262, 'reg_alpha': 0.0017073967431528124, 'max_bin': 128}. Best is trial 0 with value: 0.2916862987004084.
[I 2026-05-10 09:21:58,813] Trial 1 finished with value: 0.28901393920059165 and parameters: {'max_depth': 6, 'min_child_weight': 1.0636066512540283, 'learning_rate': 0.139959090366385, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'reg_lambda': 0.006517070593249451, 'reg_alpha': 0.00541524411940254, 'max_bin': 256}. Best is trial 0 with value: 0.2916862987004084.
[I 2026-05-10 09:21:59,569] Trial 2 finished with value: 0.29782947813141913 and parameters: {'max_depth': 5, 'min_child_weight': 2.3927528765580632, 'learning_rate': 0.06136830861665677, 'subsample': 0.6557975

{'max_depth': 3,
 'min_child_weight': 1.040263412882208,
 'learning_rate': 0.10156987104839743,
 'subsample': 0.8989576749220837,
 'colsample_bytree': 0.8259004709914184,
 'reg_lambda': 0.34927838978162373,
 'reg_alpha': 0.0016547670699918167,
 'max_bin': 256}

## 8. Train final model tren train set

In [21]:
best_params = {
    **base_params,
    **study.best_params,
    "n_estimators": FINAL_N_ESTIMATORS,
}

final_model = XGBClassifier(**best_params)
start = time.time()
final_model.fit(
    X_train_all,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)
print(f"Final train time: {(time.time() - start) / 60:.2f} minutes")

[0]	validation_0-aucpr:0.19145
[50]	validation_0-aucpr:0.19361
[100]	validation_0-aucpr:0.19881
[150]	validation_0-aucpr:0.19857
[200]	validation_0-aucpr:0.19772
[250]	validation_0-aucpr:0.19575
[300]	validation_0-aucpr:0.19625
[350]	validation_0-aucpr:0.19822
[400]	validation_0-aucpr:0.19733
[450]	validation_0-aucpr:0.19635
[500]	validation_0-aucpr:0.19886
[550]	validation_0-aucpr:0.19953
[600]	validation_0-aucpr:0.19946
[650]	validation_0-aucpr:0.20021
[700]	validation_0-aucpr:0.20233
[750]	validation_0-aucpr:0.20232
[800]	validation_0-aucpr:0.20247
[850]	validation_0-aucpr:0.20348
[900]	validation_0-aucpr:0.20293
[950]	validation_0-aucpr:0.20485
[1000]	validation_0-aucpr:0.20599
[1050]	validation_0-aucpr:0.20585
[1100]	validation_0-aucpr:0.20626
[1150]	validation_0-aucpr:0.20628
[1199]	validation_0-aucpr:0.20701
Final train time: 0.03 minutes


## 9. Evaluate final model on test

In [22]:
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report, confusion_matrix, precision_recall_curve

proba_val = final_model.predict_proba(X_val)[:, 1]
proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(np.int8)

metrics = {
    "val_pr_auc": average_precision_score(y_val, proba_val),
    "val_roc_auc": roc_auc_score(y_val, proba_val),
    "test_pr_auc": average_precision_score(y_test, proba_test),
    "test_roc_auc": roc_auc_score(y_test, proba_test),
    "test_accuracy_05": (pred_test == y_test).mean(),
}
print(metrics)
print(classification_report(y_test, pred_test, target_names=["benign", "pathogenic"]))

confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test, pred_test),
    index=["true_benign", "true_pathogenic"],
    columns=["pred_benign", "pred_pathogenic"],
)
confusion_matrix_df

{'val_pr_auc': 0.20908504640564882, 'val_roc_auc': 0.610021744717176, 'test_pr_auc': 0.1699664673360922, 'test_roc_auc': 0.6094595611644584, 'test_accuracy_05': np.float64(0.8736073032724727)}
              precision    recall  f1-score   support

      benign       0.88      1.00      0.93     60076
  pathogenic       0.15      0.00      0.01      8496

    accuracy                           0.87     68572
   macro avg       0.52      0.50      0.47     68572
weighted avg       0.79      0.87      0.82     68572



,pred_benign,pred_pathogenic
true_benign,59867,209
true_pathogenic,8458,38


## 10. Threshold analysis

In [23]:
precision, recall, thresholds = precision_recall_curve(y_test, proba_test)
pr_table = pd.DataFrame({
    "threshold": np.r_[thresholds, 1.0],
    "precision": precision,
    "recall": recall,
})

recall_at_precision_50 = pr_table.loc[pr_table["precision"].ge(0.50), "recall"].max()
precision_at_recall_80 = pr_table.loc[pr_table["recall"].ge(0.80), "precision"].max()

f1 = 2 * pr_table["precision"] * pr_table["recall"] / (pr_table["precision"] + pr_table["recall"] + 1e-12)
f2 = 5 * pr_table["precision"] * pr_table["recall"] / (4 * pr_table["precision"] + pr_table["recall"] + 1e-12)

best_f1_row = pr_table.iloc[f1.idxmax()].copy()
best_f1_row["f1"] = f1.max()
best_f2_row = pr_table.iloc[f2.idxmax()].copy()
best_f2_row["f2"] = f2.max()

print(f"Recall at precision >= 0.50: {recall_at_precision_50:.4f}")
print(f"Precision at recall >= 0.80: {precision_at_recall_80:.4f}")
print("Best F1 threshold:")
display(best_f1_row.to_frame().T)
print("Best F2 threshold:")
display(best_f2_row.to_frame().T)

Recall at precision >= 0.50: 0.0000
Precision at recall >= 0.80: 0.1448
Best F1 threshold:


,threshold,precision,recall,f1
1800,0.103418,0.17341,0.499411,0.257432


Best F2 threshold:


,threshold,precision,recall,f2
413,0.040386,0.133797,0.921963,0.423277


## 11. Save outputs

In [24]:
import joblib

model_path = MODEL_DIR / "clinvar_xgboost_optuna_tuned.joblib"
prediction_path = PROCESSED_DIR / "xgboost_optuna_tuned_test_predictions.parquet"
threshold_path = PROCESSED_DIR / "xgboost_optuna_tuned_threshold_table.parquet"
study_path = PROCESSED_DIR / "xgboost_optuna_tuned_trials.parquet"

prediction_df = df.iloc[test_idx][[
    "GeneSymbol", "ClinicalSignificance", "Y", "Chromosome", "PositionVCF",
    "ReferenceAlleleVCF", "AlternateAlleleVCF", "ReviewStatus", "ReviewStatusSimple",
    "OriginSimple", "NumberSubmitters_int", "PhenotypeList",
]].copy()
prediction_df["pred_proba_pathogenic"] = proba_test
prediction_df["pred_label_05"] = pred_test

bundle = {
    "preprocessor": preprocessor,
    "model": final_model,
    "feature_columns": feature_df.columns.tolist(),
    "categorical_cols": categorical_cols,
    "numeric_cols": numeric_cols,
    "best_params": best_params,
    "metrics": metrics,
}
joblib.dump(bundle, model_path)
prediction_df.to_parquet(prediction_path, index=False, engine="pyarrow")
pr_table.to_parquet(threshold_path, index=False, engine="pyarrow")
study.trials_dataframe().to_parquet(study_path, index=False, engine="pyarrow")

print(f"Saved model bundle: {model_path}")
print(f"Saved predictions: {prediction_path}")
print(f"Saved thresholds: {threshold_path}")
print(f"Saved Optuna trials: {study_path}")

Saved model bundle: /mnt/MyData/Bioinformatics/Project/models/clinvar_xgboost_optuna_tuned.joblib
Saved predictions: /mnt/MyData/Bioinformatics/Project/processed/xgboost_optuna_tuned_test_predictions.parquet
Saved thresholds: /mnt/MyData/Bioinformatics/Project/processed/xgboost_optuna_tuned_threshold_table.parquet
Saved Optuna trials: /mnt/MyData/Bioinformatics/Project/processed/xgboost_optuna_tuned_trials.parquet
